# Project 5 — Real-Time Voice Assistant

Pipeline: **audio → ASR (Deepgram) → LLM (OpenAI) → TTS (ElevenLabs)**. Self-contained **replay mode** — no mic needed: it synthesizes a sample question, then runs the full pipeline and measures the **latency budget**. Emits **`RESULTS.md`**.

Needs in `../.env`: `DEEPGRAM_API_KEY`, `OPENAI_API_KEY`, `ELEVENLABS_API_KEY`.

1. End-to-end streaming pipeline working
2. Latency budget (ASR / LLM TTFT / TTS TTFB / overhead)
3. Resilience: timeouts, graceful degradation, replay mode

In [ ]:
%pip install -q requests openai python-dotenv

In [ ]:
import os, time, requests
from dotenv import load_dotenv
load_dotenv('../.env')
DG = os.getenv('DEEPGRAM_API_KEY'); EL = os.getenv('ELEVENLABS_API_KEY')
assert os.getenv('OPENAI_API_KEY') and DG and EL, 'Set OPENAI_API_KEY, DEEPGRAM_API_KEY, ELEVENLABS_API_KEY in ../.env'
from openai import OpenAI
oai = OpenAI()
VOICE_ID = '21m00Tcm4TlvDq8ikWAM'  # ElevenLabs default 'Rachel'

### Build a sample input (replay mode)
Synthesize a spoken question so the notebook runs without a microphone. Drop your own `input.wav` here to use real audio instead.

In [ ]:
SAMPLE_QUESTION = 'What is retrieval augmented generation, in one sentence?'

def tts(text, path):
    """ElevenLabs TTS -> mp3 file. Returns time-to-first-byte."""
    url = f'https://api.elevenlabs.io/v1/text-to-speech/{VOICE_ID}'
    headers = {'xi-api-key': EL, 'Content-Type': 'application/json'}
    body = {'text': text, 'model_id': 'eleven_turbo_v2_5'}
    t0 = time.perf_counter(); ttfb = None
    with requests.post(url, json=body, headers=headers, stream=True, timeout=30) as r:
        r.raise_for_status()
        with open(path, 'wb') as f:
            for chunk in r.iter_content(4096):
                if ttfb is None: ttfb = time.perf_counter() - t0
                f.write(chunk)
    return ttfb

if not os.path.exists('input.wav'):
    tts(SAMPLE_QUESTION, 'input.wav')  # mp3 bytes; Deepgram detects container
print('input ready:', os.path.getsize('input.wav'), 'bytes')

## Phase 1 — Pipeline end-to-end

In [ ]:
def asr(path):
    """Deepgram prerecorded transcription. Returns (transcript, latency_s)."""
    t0 = time.perf_counter()
    r = requests.post('https://api.deepgram.com/v1/listen?model=nova-2&smart_format=true',
                      headers={'Authorization': f'Token {DG}', 'Content-Type': 'audio/*'},
                      data=open(path, 'rb').read(), timeout=30)
    r.raise_for_status()
    txt = r.json()['results']['channels'][0]['alternatives'][0]['transcript']
    return txt, time.perf_counter() - t0

def llm_stream(prompt):
    """OpenAI streaming. Returns (full_text, time_to_first_token_s)."""
    t0 = time.perf_counter(); ttft = None; parts = []
    stream = oai.chat.completions.create(model='gpt-4o-mini', stream=True,
        messages=[{'role':'system','content':'Answer in one concise sentence.'},
                  {'role':'user','content':prompt}])
    for ev in stream:
        d = ev.choices[0].delta.content
        if d:
            if ttft is None: ttft = time.perf_counter() - t0
            parts.append(d)
    return ''.join(parts), ttft

transcript, asr_s = asr('input.wav')
print('ASR:', transcript)
reply, llm_ttft = llm_stream(transcript)
print('LLM:', reply)
tts_ttfb = tts(reply, 'output.wav')
print('spoke reply -> output.wav')

## Phase 2 — Latency budget
Decompose end-to-end response time into ASR / LLM-TTFT / TTS-TTFB / overhead, averaged over runs.

In [ ]:
def one_turn():
    t0 = time.perf_counter()
    txt, a = asr('input.wav')
    reply, l = llm_stream(txt)
    t = tts(reply, 'output.wav')
    total = time.perf_counter() - t0
    return {'asr_s': a, 'llm_ttft_s': l, 'tts_ttfb_s': t,
            'overhead_s': total - (a + l + t), 'total_s': total}

import statistics
runs = [one_turn() for _ in range(3)]
budget = {k: round(statistics.mean(r[k] for r in runs), 3) for k in runs[0]}
for k, v in budget.items():
    print(f'{k:14s} {v:.3f}s')

## Phase 3 — Resilience
Timeout handling + graceful degradation when a stage fails.

In [ ]:
def robust_turn(path, asr_timeout=10, llm_timeout=15):
    """Each stage guarded; degrade gracefully instead of hanging."""
    try:
        txt, _ = asr(path)
    except Exception as e:
        return {'status': 'asr_failed', 'spoken': 'Sorry, I could not hear that. Please try again.', 'error': str(e)}
    if not txt.strip():
        return {'status': 'empty_transcript', 'spoken': 'I did not catch that.'}
    try:
        reply, _ = llm_stream(txt)
    except Exception as e:
        return {'status': 'llm_failed', 'spoken': 'I am having trouble thinking right now.', 'error': str(e)}
    try:
        tts(reply, 'output.wav'); status = 'ok'
    except Exception:
        status = 'tts_failed_text_fallback'  # degrade to text
    return {'status': status, 'transcript': txt, 'reply': reply}

# Replay mode: feed recorded input back through for debugging
print(robust_turn('input.wav'))
print(robust_turn('does_not_exist.wav'))  # demonstrates graceful ASR failure

In [ ]:
# Write RESULTS.md
bar = lambda s: '█' * max(1, round(s / budget['total_s'] * 30))
md = ['# Project 5 — Voice Assistant Latency Report', '',
      f'Track: voice (Deepgram ASR + GPT-4o-mini + ElevenLabs TTS). Replay mode, mean of {len(runs)} runs.',
      f'Sample question: "{SAMPLE_QUESTION}"', '',
      '## Latency Budget', '', '| Stage | Seconds | Share |', '|---|---|---|']
for k in ('asr_s','llm_ttft_s','tts_ttfb_s','overhead_s'):
    md.append(f'| {k} | {budget[k]:.3f} | {bar(budget[k])} |')
md += [f'| **total_s** | **{budget["total_s"]:.3f}** | |', '',
       '## Resilience', '',
       '- Per-stage try/except with graceful spoken fallbacks',
       '- Timeouts on every network call (no indefinite hang)',
       '- Replay mode: recorded `input.wav` re-fed through the pipeline for debugging',
       '- TTS failure degrades to text instead of crashing']
with open('RESULTS.md', 'w') as f:
    f.write('\n'.join(md))
print('Wrote RESULTS.md')